In [1]:
import pandas as pd
import pickle
import numpy as np
import torch
from sklearn.neighbors import NearestNeighbors
from torch_geometric.data import HeteroData
import torch.nn as nn
from torch_geometric.nn import HeteroConv, ChebConv
import plotly.graph_objects as go
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.sparse as sp
import torch.nn.functional as F

c:\Users\lucch\Desktop\thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# SECTION 1: IMPORTS & INITIAL RAW DATA LOADING
# ==========================================
with open("processed_data_pkl/google_master.pkl", "rb") as f:
    google = pickle.load(f)

with open("processed_data_pkl/pollution_dic.pkl", "rb") as f:
    pollution = pickle.load(f)

with open("processed_data_pkl/processed_traffic_data.pkl", "rb") as f:
    traffic = pickle.load(f)

In [3]:
for site_ref, directions in google.items():
    for direction, df in directions.items():
        agg_dict = {"Count": "sum"}
        for avg_col in ["speed(mph)", "miles", "avtime", "TravelTime"]:
            if avg_col in df.columns:
                agg_dict[avg_col] = "mean"

        google[site_ref][direction] = (
            df
            .assign(timestamp=df['timestamp'].dt.floor('1h'))
            .groupby('timestamp', as_index=False)
            .agg(agg_dict)
        )


In [4]:
for site_ref, directions in traffic.items():
    for direction, df in directions.items():
        if site_ref in google and direction in google[site_ref]:
            add_df = google[site_ref][direction][["timestamp"] + [c for c in ["speed(mph)", "miles", "avtime", "TravelTime"] if c in google[site_ref][direction].columns]].copy()
            if not add_df.empty:
                traffic[site_ref][direction] = (
                    df.merge(add_df, on="timestamp", how="left")
                )


In [5]:
google_speed = pd.read_csv("classified_datasets/traffic/google_speed/google1.csv")
google_speed["section"] = google_speed["section"].apply(lambda x: x.split("-")[1]).astype(int)
google_speed

,section,timestamp,miles,avtime,TravelTime,speed(mph)
0,1,2017-07-02 22:48:12,9.411907,0.457222,0.389444,24.167520
1,2,2017-07-02 22:48:13,4.449638,0.236944,0.201111,22.125270
2,3,2017-07-02 22:48:13,5.587989,0.411667,0.339167,16.475644
3,4,2017-07-02 22:48:14,4.315422,0.230833,0.200556,21.517338
4,5,2017-07-02 22:48:14,4.611816,0.309444,0.251389,18.345344
...,...,...,...,...,...,...
552028,17,2018-04-18 10:30:10,11.891177,0.498056,0.500000,23.782354
552029,18,2018-04-18 10:30:10,10.619852,0.496389,0.488889,21.722424
552030,19,2018-04-18 10:30:11,11.999295,0.531667,0.538333,22.289713
552031,20,2018-04-18 10:30:11,19.283628,0.695278,0.662778,29.095163


In [6]:
google_speed["timestamp"] = pd.to_datetime(google_speed["timestamp"])
google_speed = google_speed.sort_values(["section", "timestamp"]).reset_index(drop=True)

google_speed_sections = {}

for section, group in google_speed.groupby("section", sort=True):
    group = group.sort_values("timestamp").reset_index(drop=True)
    hour_index = pd.date_range(
        start=group["timestamp"].min().floor("h"),
        end=group["timestamp"].max().ceil("h"),
        freq="h"
    )
    targets = pd.DataFrame({"timestamp": hour_index})
    nearest_hourly = pd.merge_asof(
        targets,
        group,
        on="timestamp",
        direction="nearest"
    )
    google_speed_sections[section] = nearest_hourly

google_speed_sections

{1:                timestamp  section      miles    avtime  TravelTime  speed(mph)
 0    2017-07-02 22:00:00        1   9.411907  0.457222    0.389444   24.167520
 1    2017-07-02 23:00:00        1   9.411907  0.457222    0.388889   24.202045
 2    2017-07-03 00:00:00        1   9.411907  0.457222    0.377778   24.913870
 3    2017-07-03 01:00:00        1   9.411907  0.457222    0.389722   24.150295
 4    2017-07-03 02:00:00        1   9.411907  0.457222    0.394444   23.861172
 ...                  ...      ...        ...       ...         ...         ...
 6945 2018-04-18 07:00:00        1   9.420606  0.470000    0.428889   21.965143
 6946 2018-04-18 08:00:00        1   9.411907  0.474167    0.586944   16.035430
 6947 2018-04-18 09:00:00        1   9.418120  0.508056    0.590833   15.940401
 6948 2018-04-18 10:00:00        1  10.663348  0.479722    0.483056   22.074785
 6949 2018-04-18 11:00:00        1  10.663348  0.479722    0.487500   21.873534
 
 [6950 rows x 6 columns],
 2:      

In [7]:
for sensor, directions in traffic.items():
    for direction, df in directions.items():
        cols_to_drop = [c for c in df.columns if c in ["speed(mph)", "miles", "avtime", "TravelTime"]]
        df = df.drop(columns=cols_to_drop, errors="ignore")

        if df.empty:
            continue

        route = df.loc[0, "route"]
        if route not in google_speed_sections:
            continue

        gdf = google_speed_sections[route]

        # --- Build full timestamp index (correct way) ---
        full_index = pd.Index(df["timestamp"]).union(pd.Index(gdf["timestamp"]))

        # --- Expand traffic DF to full timeline ---
        expanded = (
            df.set_index("timestamp")
              .reindex(full_index)
              .reset_index()
              .rename(columns={"index": "timestamp"})
        )

        # --- Merge Google speed data cleanly ---
        merged = expanded.merge(
            gdf[["timestamp", "speed(mph)", "miles", "avtime", "TravelTime"]],
            on="timestamp",
            how="left"
        )

        traffic[sensor][direction] = merged

In [8]:
master = pd.Index(traffic[2]["N"]["timestamp"])
for sensor, directions in traffic.items():
    for direction, df in directions.items():
        if len(df.columns)<10:
            continue
        # Reindex to the master timeline
        expanded = (
            df.set_index("timestamp")
              .reindex(master)
              .reset_index()
              .rename(columns={"index": "timestamp"})
        )

        traffic[sensor][direction] = expanded

In [9]:
for direction, df_dict in traffic.items():
    for df_id, df in df_dict.items():
        df["weekday"] = df["timestamp"].dt.weekday

        num_unique = df.nunique(dropna=True)
        single_val_cols = num_unique[num_unique == 1].index
        for col in single_val_cols:
            first_valid = df[col].dropna().iloc[0]
            df[col] = df[col].fillna(first_valid)

        # Rich temporal features — critical for learning daily traffic cycles
        df["hour"]         = df["timestamp"].dt.hour.astype(np.float32)
        df["hour_sin"]     = np.sin(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["hour_cos"]     = np.cos(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["weekday_sin"]  = np.sin(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["weekday_cos"]  = np.cos(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["is_weekend"]   = (df["weekday"] >= 5).astype(np.float32)


In [ ]:
training_data = {}
no_speed = {}
for direction, df_dict in traffic.items():
    for df_id, df in df_dict.items():
        if len(df.columns) > 16:
            if direction not in training_data:
                training_data[direction] = {}
            training_data[direction][df_id] = df
        else:
            if direction not in no_speed:
                no_speed[direction] = {}
            no_speed[direction][df_id] = df


In [11]:
# Clone full_data FIRST — before any dropna — so it keeps every row,
# including timestamps where count is NaN but speed/avtime/weekday are valid.
# Those NaN-count rows are exactly what we want to impute later.
full_data = {
    site_id: {compass_dir: df.copy(deep=True) for compass_dir, df in dir_dict.items()}
    for site_id, dir_dict in training_data.items()
}

# Add the imputation flag column (0 = observed, 1 = model-imputed)
for site_id, dir_dict in full_data.items():
    for compass_dir, df in dir_dict.items():
        df["count_imputed"] = np.int8(0)

# Now strip NaN-count rows from training_data only —
# the supervised loss needs real observed targets.
for site_id, dir_dict in training_data.items():
    for compass_dir, df in dir_dict.items():
        df.dropna(subset=["count"], inplace=True)


In [12]:
import pandas as pd

for direction, df_dict in training_data.items():
    for df_id, df in df_dict.items():

        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp")

        start = df["timestamp"].min()
        full_index = pd.date_range(start=start, periods=336, freq="1h")

        df = df.set_index("timestamp").reindex(full_index)

        # Fill non-temporal numeric columns (speed, count, etc.) with last-week shift then ffill
        temporal_cols = {"hour", "hour_sin", "hour_cos",
                         "weekday", "weekday_sin", "weekday_cos", "is_weekend"}
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        fill_cols = [c for c in numeric_cols if c not in temporal_cols]

        missing_mask = df[fill_cols].isna().any(axis=1)
        df.loc[missing_mask, fill_cols] = df[fill_cols].shift(336).loc[missing_mask]
        df[fill_cols] = df[fill_cols].ffill().bfill()

        df = df.reset_index().rename(columns={"index": "timestamp"})

        # Recompute temporal features from the actual timestamp — ffill would give wrong values
        df["hour"]        = df["timestamp"].dt.hour.astype(np.float32)
        df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24).astype(np.float32)
        df["weekday"]     = df["timestamp"].dt.weekday.astype(np.float32)
        df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7).astype(np.float32)
        df["is_weekend"]  = (df["timestamp"].dt.weekday >= 5).astype(np.float32)

        training_data[direction][df_id] = df


In [13]:
# ==========================================
# SECTION 2: IMPROVED SPATIAL ALIGNMENT & HETERO DATA CONSTRUCTION
# ==========================================

def build_hetero_node_records(training_data, pollution_data, traffic_periods=336, traffic_freq="1h"):
    """Build unified node records with improved feature extraction from concat."""
    traffic_records = {}
    pollution_records = {}
    traffic_frames = []

    # Process traffic sensors as unique directional keys
    for direction, df_dict in training_data.items():
        for df_id, df in df_dict.items():
            node_key = f"traffic_{df_id}_{direction}"
            df_copy = df.copy()
            df_copy["timestamp"] = pd.to_datetime(df_copy["timestamp"])
            df_copy = df_copy.sort_values("timestamp").set_index("timestamp")

            lon = df_copy["longitude"].iloc[0]
            lat = df_copy["latitude"].iloc[0]

            traffic_records[node_key] = {"df": df_copy, "lon": lon, "lat": lat, "type": "traffic"}
            traffic_frames.append(df_copy)

    if not traffic_frames:
        raise ValueError("No traffic data frames found.")

    traffic_start = min(frame.index.min() for frame in traffic_frames)
    master_index = pd.date_range(start=traffic_start, periods=traffic_periods, freq=traffic_freq)

    # Process pollution nodes
    for site, df in pollution_data.items():
        node_key = f"pollution_{site}"
        df_copy = df.copy()
        df_copy["timestamp"] = pd.to_datetime(df_copy["timestamp"])
        df_copy = df_copy.sort_values("timestamp").set_index("timestamp")

        # Check overlap with master index
        if len(master_index.intersection(df_copy.index)) == 0:
            continue

        lon_col = "long" if "long" in df_copy.columns else "longitude"
        lat_col = "lat" if "lat" in df_copy.columns else "latitude"
        lon = float(df_copy[lon_col].iloc[0])
        lat = float(df_copy[lat_col].iloc[0])

        # Some pollution data files have columns named long/lat but values stored in reversed order.
        if abs(lon) > 10 and abs(lat) <= 10:
            lon, lat = lat, lon

        pollution_records[node_key] = {"df": df_copy, "lon": lon, "lat": lat, "type": "pollution"}

    # Find common timestamps across all nodes
    common_timestamps = set(master_index)
    for k, r in traffic_records.items():
        common_timestamps = common_timestamps.intersection(r["df"].index)
    for k, r in pollution_records.items():
        common_timestamps = common_timestamps.intersection(r["df"].index)

    aligned_timestamps = sorted(list(common_timestamps))
    if not aligned_timestamps:
        raise ValueError("No overlapping timestamps found across traffic and pollution sets.")
        
    return traffic_records, pollution_records, aligned_timestamps, master_index

# Execute alignment step
traffic_records, pollution_records, aligned_timestamps, master_index = build_hetero_node_records(training_data=training_data, pollution_data=pollution)
aligned_index = pd.DatetimeIndex(aligned_timestamps)

traffic_nodes = sorted(list(traffic_records.keys()))
pollution_nodes = sorted(list(pollution_records.keys()))

print(f"Traffic nodes: {len(traffic_nodes)}")
print(f"Pollution nodes: {len(pollution_nodes)}")
print(f"Aligned timestamps: {len(aligned_timestamps)}")
print(f"Sample traffic node: {traffic_nodes[0] if traffic_nodes else 'None'}")
print(f"Sample pollution node: {pollution_nodes[0] if pollution_nodes else 'None'}")


# Extract all numeric features and align them using the shared overlap timeline
def extract_aligned_features(
    records,
    master_index,
    domain_name="traffic",
    never_fill=(),
    mask_count_missing=False,
    exclude_from_input=(),
):
    """
    Extract aligned feature matrix from node records.

    Returns X [T,N,F], feature_names, node_ids, y_count [T,N] or None.
    Lagged count features (lag1/2/3/24) are added automatically for traffic nodes
    so the model has temporal context without leaking the current-step target.
    """
    aligned_data = {}
    never_fill = set(never_fill)
    exclude_from_input = set(exclude_from_input)

    for node_id, rec in records.items():
        df = rec["df"].reindex(master_index)
        numeric_df = df.select_dtypes(include=[np.number]).copy()
        numeric_df = numeric_df.drop(
            columns=["longitude", "latitude", "long", "lat", "sensor_id", "count_imputed"],
            errors="ignore",
        )
        if mask_count_missing and domain_name == "traffic" and "count" in numeric_df.columns:
            numeric_df["count_observed"] = (~numeric_df["count"].isna()).astype(np.float32)
        aligned_data[node_id] = numeric_df

    # Collect feature names before exclusion
    all_feature_names = []
    for numeric_df in aligned_data.values():
        for col in numeric_df.columns.tolist():
            if col not in all_feature_names:
                all_feature_names.append(col)

    T = len(master_index)
    N = len(aligned_data)
    node_ids = list(aligned_data.keys())

    X_full = np.full((T, N, len(all_feature_names)), np.nan, dtype=np.float32)
    for i, node_id in enumerate(node_ids):
        numeric_df = aligned_data[node_id]
        for j, feature in enumerate(all_feature_names):
            if feature in numeric_df.columns:
                X_full[:, i, j] = numeric_df[feature].values

    # Forward/back fill (skip count_observed and never_fill)
    skip_fill = never_fill | {"count_observed"}
    for i in range(N):
        for j, fname in enumerate(all_feature_names):
            if fname in skip_fill:
                continue
            col = X_full[:, i, j]
            mask = np.isnan(col)
            if not mask.any():
                continue
            for t in range(1, T):
                if np.isnan(col[t]) and not np.isnan(col[t - 1]):
                    col[t] = col[t - 1]
            for t in range(T - 2, -1, -1):
                if np.isnan(col[t]) and not np.isnan(col[t + 1]):
                    col[t] = col[t + 1]
            X_full[:, i, j] = col

    # Extract y_count before any exclusion
    y_count = None
    if domain_name == "traffic" and "count" in all_feature_names:
        count_j = all_feature_names.index("count")
        y_count = X_full[:, :, count_j].copy()  # [T, N]

    # Add lagged count features for traffic domain
    # These are NOT excluded — they give the model temporal context
    if domain_name == "traffic" and "count" in all_feature_names:
        count_j = all_feature_names.index("count")
        count_col = X_full[:, :, count_j]  # [T, N]
        lag_configs = [("count_lag1", 1), ("count_lag2", 2),
                       ("count_lag3", 3), ("count_lag24", 24)]
        lag_arrays = []
        lag_names  = []
        for lag_name, lag in lag_configs:
            lagged = np.full_like(count_col, np.nan)
            lagged[lag:] = count_col[:-lag]
            # forward/back fill each lag
            for i in range(N):
                col = lagged[:, i]
                mask = np.isnan(col)
                if mask.any():
                    for t in range(1, T):
                        if np.isnan(col[t]) and not np.isnan(col[t - 1]):
                            col[t] = col[t - 1]
                    for t in range(T - 2, -1, -1):
                        if np.isnan(col[t]) and not np.isnan(col[t + 1]):
                            col[t] = col[t + 1]
                    lagged[:, i] = col
            lag_arrays.append(lagged[:, :, np.newaxis])
            lag_names.append(lag_name)

        X_full = np.concatenate([X_full] + lag_arrays, axis=2)
        all_feature_names = all_feature_names + lag_names

    # Build input array, dropping excluded columns
    input_feature_names = [f for f in all_feature_names if f not in exclude_from_input]
    input_indices = [all_feature_names.index(f) for f in input_feature_names]
    X = X_full[:, :, input_indices]

    return X, input_feature_names, node_ids, y_count


# Extract features for training — count excluded from X (it is the target)
# Lagged count features remain in X and give temporal context
X_traffic_raw, traffic_features, traffic_node_ids, y_count_train = extract_aligned_features(
    traffic_records,
    aligned_index,
    "traffic",
    never_fill=(),
    mask_count_missing=True,
    exclude_from_input=("count",),
)
X_pollution_raw, pollution_features, pollution_node_ids, _ = extract_aligned_features(
    pollution_records, aligned_index, "pollution"
)

print(f"\nTraffic features ({len(traffic_features)}): {traffic_features}")
print(f"Pollution features ({len(pollution_features)}): {pollution_features}")

X_traffic  = torch.tensor(X_traffic_raw,  dtype=torch.float32)
X_pollution = torch.tensor(X_pollution_raw, dtype=torch.float32)

print(f"\nX_traffic shape:  {X_traffic.shape}")
print(f"X_pollution shape: {X_pollution.shape}")
print(f"NaNs in X_traffic:  {torch.isnan(X_traffic).sum()}")
print(f"NaNs in X_pollution: {torch.isnan(X_pollution).sum()}")


Traffic nodes: 48
Pollution nodes: 4
Aligned timestamps: 332
Sample traffic node: traffic_E_11
Sample pollution node: pollution_guildhall

Traffic features (19): ['route', 'weekday', 'distance_to_center_km', 'traffic_intensity', 'speed(mph)', 'miles', 'avtime', 'TravelTime', 'hour', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'count_observed', 'count_lag1', 'count_lag2', 'count_lag3', 'count_lag24']
Pollution features (3): ['NOx', 'NO', 'NO2']

X_traffic shape:  torch.Size([332, 48, 19])
X_pollution shape: torch.Size([332, 4, 3])
NaNs in X_traffic:  0
NaNs in X_pollution: 0


In [14]:
coords_traffic = np.array([[traffic_records[nid]["lon"], traffic_records[nid]["lat"]] 
                           for nid in traffic_node_ids])
coords_pollution = np.array([[pollution_records[nid]["lon"], pollution_records[nid]["lat"]] 
                             for nid in pollution_node_ids])

proj_traffic = coords_traffic.copy()
proj_traffic[:, 0] = proj_traffic[:, 0]

proj_pollution = coords_pollution.copy()
proj_pollution[:, 0] = proj_pollution[:, 0]


def build_intra_domain_edges_robust(coords, k=6):
    """Builds a connected graph within a domain using symmetrized k-NN."""
    A = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(coords).kneighbors_graph(mode='connectivity')
    
    A_coo = (A + A.T).tocoo()
    mask = A_coo.row != A_coo.col
    rows, cols = A_coo.row[mask], A_coo.col[mask]
    
    return torch.tensor(np.vstack([rows, cols]), dtype=torch.long)


def build_cross_domain_edges_robust(src_coords, dst_coords, k=4):
    """
    Builds spatial bipartite edges ensuring every source node connects to destinations,
    while removing duplicate overlapping edge pairs.
    """
    nn = NearestNeighbors(n_neighbors=min(k, len(dst_coords)), metric='euclidean')
    nn.fit(dst_coords)
    idx = nn.kneighbors(src_coords, return_distance=False)
    
    src_indices = np.repeat(np.arange(len(src_coords)), idx.shape[1])
    dst_indices = idx.flatten()
    
    # Clear out any accidental redundant edge indexes
    edges = np.vstack([src_indices, dst_indices])
    unique_edges = np.unique(edges, axis=1)
    
    return torch.tensor(unique_edges, dtype=torch.long)


edge_t_to_t = build_intra_domain_edges_robust(proj_traffic, k=6)
edge_p_to_t = build_cross_domain_edges_robust(proj_pollution, proj_traffic, k=6)


print(f"Symmetrized Traffic-to-Traffic Edges: {edge_t_to_t.shape}")
print(f"Bipartite Pollution-to-Traffic Edges: {edge_p_to_t.shape}")

Symmetrized Traffic-to-Traffic Edges: torch.Size([2, 380])
Bipartite Pollution-to-Traffic Edges: torch.Size([2, 24])


In [15]:
def visualize_hetero_spatial_graph(
    coords_traffic, 
    coords_pollution, 
    edge_t_to_t, 
    edge_p_to_t, 
    mapbox_style="carto-positron"
):
    """
    Visualizes intra-domain (traffic-to-traffic) and cross-domain (pollution-to-traffic) 
    spatial edges directly on a map using Plotly Scattermapbox.
    """
    fig = go.Figure()

    # --- 1. DRAW INTRA-DOMAIN EDGES (Traffic to Traffic) ---
    t_edge_lon, t_edge_lat = [], []
    t_start_nodes = edge_t_to_t[0].numpy()
    t_end_nodes = edge_t_to_t[1].numpy()
    
    for src, dst in zip(t_start_nodes, t_end_nodes):
        t_edge_lon.extend([coords_traffic[src, 0], coords_traffic[dst, 0], None])
        t_edge_lat.extend([coords_traffic[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=t_edge_lon, lat=t_edge_lat,
        mode='lines',
        line=dict(width=1.5, color='rgba(50, 150, 250, 0.4)'),
        name='Traffic-to-Traffic Edges',
        hoverinfo='none'
    ))

    # --- 2. DRAW CROSS-DOMAIN EDGES (Pollution to Traffic) ---
    p2t_edge_lon, p2t_edge_lat = [], []
    p_start_nodes = edge_p_to_t[0].numpy()
    t_end_nodes = edge_p_to_t[1].numpy()
    
    for src, dst in zip(p_start_nodes, t_end_nodes):
        p2t_edge_lon.extend([coords_pollution[src, 0], coords_traffic[dst, 0], None])
        p2t_edge_lat.extend([coords_pollution[src, 1], coords_traffic[dst, 1], None])
        
    fig.add_trace(go.Scattermapbox(
        lon=p2t_edge_lon, lat=p2t_edge_lat,
        mode='lines',
        line=dict(width=2, color='rgba(230, 90, 90, 0.6)'),
        name='Pollution-to-Traffic Edges',
        hoverinfo='none'
    ))

    # --- 3. DRAW TRAFFIC NODES ---
    fig.add_trace(go.Scattermapbox(
        lon=coords_traffic[:, 0], lat=coords_traffic[:, 1],
        mode='markers',
        marker=dict(size=9, color='blue', opacity=0.85),
        name='Traffic Sensor Nodes',
        text=[f"Traffic ID: {i}" for i in range(len(coords_traffic))],
        hoverinfo='text'
    ))

    # --- 4. DRAW POLLUTION NODES ---
    fig.add_trace(go.Scattermapbox(
        lon=coords_pollution[:, 0], lat=coords_pollution[:, 1],
        mode='markers',
        marker=dict(size=12, color='darkred', symbol='circle', opacity=0.9),
        name='Pollution Monitor Nodes',
        text=[f"Pollution ID: {i}" for i in range(len(coords_pollution))],
        hoverinfo='text'
    ))

    # --- 5. MAP LAYOUT CONFIGURATION ---
    center_lat = np.mean(np.concatenate([coords_traffic[:, 1], coords_pollution[:, 1]]))
    center_lon = np.mean(np.concatenate([coords_traffic[:, 0], coords_pollution[:, 0]]))

    fig.update_layout(
        title=dict(text='Heterogeneous Spatio-Temporal Graph Topology', font=dict(size=18)),
        autosize=True,
        hovermode='closest',
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.7)"),
        mapbox=dict(
            style=mapbox_style,
            bearing=0,
            center=dict(lat=center_lat, lon=center_lon),
            pitch=0,
            zoom=11
        ),
        width=1100,
        height=750,
        margin=dict(r=0, t=40, l=0, b=0)
    )
    
    fig.show()

visualize_hetero_spatial_graph(
    coords_traffic=coords_traffic,
    coords_pollution=coords_pollution,
    edge_t_to_t=edge_t_to_t,
    edge_p_to_t=edge_p_to_t
)

C:\Users\lucch\AppData\Local\Temp\ipykernel_20268\3585052928.py:23: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\lucch\AppData\Local\Temp\ipykernel_20268\3585052928.py:40: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\lucch\AppData\Local\Temp\ipykernel_20268\3585052928.py:49: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
C:\Users\lucch\AppData\Local\Temp\ipykernel_20268\3585052928.py:59: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


In [16]:
# ==========================================
# SECTION 3: SPATIO-TEMPORAL HETERO DATASET WITH PROPER SPLITTING
# ==========================================

class HeteroSTGDataset(torch.utils.data.Dataset):
    def __init__(self, X_traffic, X_pollution, y_count, seq_len=12):
        """
        Parameters
        ----------
        X_traffic  : [T, N_t, F_t]  model input features (count excluded)
        X_pollution: [T, N_p, F_p]
        y_count    : [T, N_t]       raw count values — used as prediction targets
        seq_len    : input sequence length
        """
        self.X_traffic  = X_traffic
        self.X_pollution = X_pollution
        self.y_count    = y_count      # [T, N_t]  — may contain NaN for missing
        self.seq_len    = seq_len

    def __len__(self):
        return self.X_traffic.shape[0] - self.seq_len

    def __getitem__(self, idx):
        x_traffic_seq  = self.X_traffic[idx : idx + self.seq_len]       # [Seq, N_t, F_t]
        x_pollution_seq = self.X_pollution[idx : idx + self.seq_len]    # [Seq, N_p, F_p]
        y_target = self.y_count[idx + self.seq_len]                     # [N_t]
        return x_traffic_seq, x_pollution_seq, y_target


# y_count_train comes from extract_aligned_features (before count was excluded from X)
y_count_tensor_train = torch.tensor(y_count_train, dtype=torch.float32)  # [T, N_t]

dataset = HeteroSTGDataset(X_traffic, X_pollution, y_count_tensor_train, seq_len=12)

print(f"Total sequences available: {len(dataset)}")
print(f"Input traffic features ({len(traffic_features)}): {traffic_features}")

# Split: 70% train, 15% val, 15% test (temporal — no shuffle across splits)
total_samples = len(dataset)
train_frac = 0.7
val_frac   = 0.15

train_end = int(total_samples * train_frac)
val_end   = train_end + int(total_samples * val_frac)

train_indices = list(range(train_end))
val_indices   = list(range(train_end, val_end))
test_indices  = list(range(val_end, total_samples))

print(f"Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}")

from torch.utils.data import DataLoader, Subset

batch_size = 16
train_set = Subset(dataset, train_indices)
val_set   = Subset(dataset, val_indices)
test_set  = Subset(dataset, test_indices)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


Total sequences available: 320
Input traffic features (19): ['route', 'weekday', 'distance_to_center_km', 'traffic_intensity', 'speed(mph)', 'miles', 'avtime', 'TravelTime', 'hour', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'count_observed', 'count_lag1', 'count_lag2', 'count_lag3', 'count_lag24']
Train: 224 | Val: 48 | Test: 48
Train batches: 14 | Val batches: 3 | Test batches: 3


In [17]:
# ==========================================
# BUILD FULL_DATA INPUTS FOR LATER IMPUTATION
# ==========================================

full_data_traffic_records, _, _, _ = build_hetero_node_records(
    training_data=full_data,
    pollution_data=pollution
)
# Build aligned_index_full as the full hourly range covering all traffic records
_all_starts = [r["df"].index.min() for r in full_data_traffic_records.values()]
_all_ends   = [r["df"].index.max() for r in full_data_traffic_records.values()]
aligned_index_full = pd.date_range(
    start=min(_all_starts), end=max(_all_ends), freq="1h"
)

# For imputation we include count in X (observed values provide context).
# count_observed=0 tells the model which slots are missing and need filling.
# count is forward-filled so the model sees the last known value as context.
X_traffic_full_raw, traffic_features_full, traffic_node_ids_full, y_count_full = extract_aligned_features(
    full_data_traffic_records, aligned_index_full, "traffic",
    never_fill=(),
    mask_count_missing=True,
    exclude_from_input=("count",),  # must match training feature set (same F_t for the model)
)
X_pollution_full_raw, pollution_features_full, pollution_node_ids_full, _ = extract_aligned_features(
    pollution_records, aligned_index_full, "pollution"
)

X_traffic_full = torch.tensor(X_traffic_full_raw, dtype=torch.float32)
X_pollution_full = torch.tensor(X_pollution_full_raw, dtype=torch.float32)

# Sanity check: NaN in count column means imputation needed
count_idx_full = traffic_features_full.index("count") if "count" in traffic_features_full else -1
if count_idx_full >= 0:
    n_nan_count = int(torch.isnan(X_traffic_full[:, :, count_idx_full]).sum())
    n_total = X_traffic_full[:, :, count_idx_full].numel()
    print(f"Count NaNs in full dataset: {n_nan_count:,} / {n_total:,} ({100*n_nan_count/n_total:.1f}%) — these will be imputed")

# Pre-compute edge arrays (CPU only — edge_index_full tensor built later after device is set)
coords_traffic_full = np.array([[full_data_traffic_records[nid]["lon"], full_data_traffic_records[nid]["lat"]]
                                for nid in traffic_node_ids_full])
N_t_full = len(traffic_node_ids_full)
N_p_full = len(pollution_node_ids_full)

edge_t_to_t_full = build_intra_domain_edges_robust(coords_traffic_full, k=min(6, N_t_full - 1))
edge_p_to_t_full = build_cross_domain_edges_robust(coords_pollution, coords_traffic_full, k=min(6, N_t_full))

edge_p_to_t_full_np = edge_p_to_t_full.cpu().numpy().copy()
edge_p_to_t_full_np[0, :] += N_t_full
all_edges_full = np.concatenate([edge_t_to_t_full.cpu().numpy(), edge_p_to_t_full_np], axis=1)
# NOTE: edge_index_full is created in the imputation setup cell below, once device is defined

y_count_tensor_full = torch.tensor(y_count_full, dtype=torch.float32)  # [T, N_t_full]
imputation_dataset = HeteroSTGDataset(
    X_traffic_full, X_pollution_full, y_count_tensor_full, seq_len=12
)
print(f"Built full_data imputation dataset: {X_traffic_full.shape}, {X_pollution_full.shape}")


Built full_data imputation dataset: torch.Size([6950, 48, 19]), torch.Size([6950, 4, 3])


In [18]:
# Verify corrected NaN/forward-fill logic
print("=" * 70)
print("VERIFICATION: Corrected forward-fill logic for count")
print("=" * 70)

# Check count values (first feature in traffic data)
count_col = X_traffic_full_raw[:, :, 0]
n_zeros = (count_col == 0).sum()
n_nonzero = (count_col > 0).sum()
n_nans = np.isnan(count_col).sum()

print(f"\nCount feature statistics:")
print(f"  Total elements:     {count_col.size:,}")
print(f"  Zero counts:        {n_zeros:,} ({100*n_zeros/count_col.size:.2f}%)")
print(f"  Non-zero counts:    {n_nonzero:,} ({100*n_nonzero/count_col.size:.2f}%)")
print(f"  NaN values:         {n_nans:,} ({100*n_nans/count_col.size:.2f}%)")
print(f"  Range: [{np.nanmin(count_col):.1f}, {np.nanmax(count_col):.1f}]")

# Check count_observed flag (should be last feature if it exists)
count_obs_idx = traffic_features_full.index("count_observed") if "count_observed" in traffic_features_full else -1
if count_obs_idx >= 0:
    count_obs_col = X_traffic_full_raw[:, :, count_obs_idx]
    n_originally_missing = (count_obs_col == 0).sum()
    n_originally_observed = (count_obs_col == 1).sum()
    print(f"\nCount_observed flag statistics:")
    print(f"  Originally observed:  {n_originally_observed:,} ({100*n_originally_observed/count_obs_col.size:.2f}%)")
    print(f"  Originally missing:   {n_originally_missing:,} ({100*n_originally_missing/count_obs_col.size:.2f}%)")

print("\nTraffic features:", traffic_features_full)
print("=" * 70)

VERIFICATION: Corrected forward-fill logic for count

Count feature statistics:
  Total elements:     333,600
  Zero counts:        0 (0.00%)
  Non-zero counts:    333,600 (100.00%)
  NaN values:         0 (0.00%)
  Range: [1.0, 20.0]

Count_observed flag statistics:
  Originally observed:  16,114 (4.83%)
  Originally missing:   317,486 (95.17%)

Traffic features: ['route', 'weekday', 'distance_to_center_km', 'traffic_intensity', 'speed(mph)', 'miles', 'avtime', 'TravelTime', 'hour', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'count_observed', 'count_lag1', 'count_lag2', 'count_lag3', 'count_lag24']


In [19]:
# ==========================================
# SECTION 4: HETEROGENEOUS ST-GCN WITH PER-NODE GRU
# ==========================================

class PerNodeGATConv(nn.Module):
    """
    Lightweight graph attention: for each node, aggregates from neighbours
    using a single-head attention weight. Avoids the N_t*H GRU blowup.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        from torch_geometric.nn import GATConv
        self.gat = GATConv(in_channels, out_channels, heads=4, concat=False,
                           dropout=0.1, add_self_loops=True)
        self.norm = nn.LayerNorm(out_channels)
        self.act  = nn.GELU()

    def forward(self, x, edge_index):
        # x: [N, C]
        return self.act(self.norm(self.gat(x, edge_index)))


class RunningStats(nn.Module):
    """
    Online mean/std normalisation with learnable scale+bias (like BatchNorm
    but uses running statistics computed from the data, not trainable mean).
    Prevents gradient blowup from raw count vs speed scale difference.
    """
    def __init__(self, num_features, momentum=0.1, eps=1e-5):
        super().__init__()
        self.momentum = momentum
        self.eps = eps
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var",  torch.ones(num_features))
        self.register_buffer("num_batches",  torch.tensor(0))
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias   = nn.Parameter(torch.zeros(num_features))

    def forward(self, x):
        # x: [..., num_features]
        if self.training:
            flat = x.detach().reshape(-1, x.shape[-1])
            mean = flat.mean(0)
            var  = flat.var(0, unbiased=False)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * var
        # x_norm computed outside if-block so it runs at eval time too
        x_norm = (x - self.running_mean) / (self.running_var + self.eps).sqrt()
        return x_norm * self.weight + self.bias


class HeteroSTGCN(nn.Module):
    """
    Heterogeneous Spatio-Temporal GCN for traffic count prediction.

    Design rationale
    ----------------
    Problem: predicting hourly traffic counts at N_t sensor nodes.
    Key insight: traffic follows strong daily/weekly cycles. The model must
    capture (1) per-node temporal dynamics and (2) spatial influence between
    nearby sensors and pollution monitors.

    Architecture
    ------------
    Input normalisation (RunningStats per domain)
        ↓
    Linear embedding (F_t → H, F_p → H)
        ↓
    [For each timestep in the sequence]
        GAT spatial encoder × 2 on unified graph (traffic + pollution)
        → outputs H-dim representation per traffic node
        ↓
    Per-node GRU  (each node has its own GRU with H-dim hidden state)
    — avoids the N_t*H blowup of the previous design
        ↓
    Final hidden state [B, N_t, H]
        + residual from time-mean of input embedding [B, N_t, H]
        ↓
    MLP head: H → H//2 → 1 → softplus
    """

    def __init__(self, N_t, N_p, F_t, F_p, hidden=64, K=3, gru_layers=2):
        super().__init__()
        self.N_t    = N_t
        self.hidden = hidden

        # Input normalisation
        self.traffic_norm   = RunningStats(F_t)
        self.pollution_norm = RunningStats(F_p)

        # Domain embeddings
        self.traffic_embed   = nn.Linear(F_t, hidden)
        self.pollution_embed = nn.Linear(F_p, hidden)

        # Spatial layers (GAT) on combined graph
        self.spatial1 = PerNodeGATConv(hidden, hidden)
        self.spatial2 = PerNodeGATConv(hidden, hidden)

        # Per-node GRU  ← key change: input_size=H, hidden_size=H  (not N_t*H)
        # We process each node independently through time
        self.gru = nn.GRU(
            input_size  = hidden,
            hidden_size = hidden,
            num_layers  = gru_layers,
            batch_first = True,
            dropout     = 0.15 if gru_layers > 1 else 0.0,
        )

        # Residual projection
        self.residual_proj = nn.Sequential(
            nn.Linear(F_t, hidden),
            nn.GELU(),
        )

        # Regression head per node
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_traffic, x_pollution, edge_index):
        """
        x_traffic  : [B, T, N_t, F_t]
        x_pollution: [B, T, N_p, F_p]
        edge_index : [2, E]
        Returns    : [B, N_t]  predicted counts
        """
        B, T, N_t, F_t = x_traffic.shape
        _,  _, N_p, _  = x_pollution.shape

        # Normalise
        xt = self.traffic_norm(x_traffic)
        xp = self.pollution_norm(x_pollution)

        # Residual from input (mean over time, before spatial mixing)
        res = self.residual_proj(xt).mean(dim=1)  # [B, N_t, H]

        # Embed
        xt = self.traffic_embed(xt)    # [B, T, N_t, H]
        xp = self.pollution_embed(xp)  # [B, T, N_p, H]

        # Spatial encoding at each timestep
        # Combine domains → apply GAT → extract traffic nodes
        spatial_out = []
        for t in range(T):
            x_t_nodes = xt[:, t]   # [B, N_t, H]
            x_p_nodes = xp[:, t]   # [B, N_p, H]
            x_combined = torch.cat([x_t_nodes, x_p_nodes], dim=1)  # [B, N, H]

            # Apply GAT per batch item (PyG GAT operates on single graphs)
            batch_out = []
            for b in range(B):
                z = x_combined[b]                      # [N, H]
                z = self.spatial1(z, edge_index)       # [N, H]
                z = self.spatial2(z, edge_index)       # [N, H]
                batch_out.append(z[:N_t])              # [N_t, H]
            spatial_out.append(torch.stack(batch_out))  # [B, N_t, H]

        # spatial_out: list of T tensors each [B, N_t, H]
        # Stack → [B, T, N_t, H]
        x_seq = torch.stack(spatial_out, dim=1)  # [B, T, N_t, H]

        # Per-node GRU: reshape to [B*N_t, T, H], run GRU, reshape back
        x_seq = x_seq.permute(0, 2, 1, 3)         # [B, N_t, T, H]
        x_seq = x_seq.reshape(B * N_t, T, -1)      # [B*N_t, T, H]

        gru_out, _ = self.gru(x_seq)               # [B*N_t, T, H]
        last = gru_out[:, -1, :]                   # [B*N_t, H]
        last = last.view(B, N_t, -1)               # [B, N_t, H]

        # Residual + head
        last = last + res
        out  = self.head(last).squeeze(-1)         # [B, N_t]
        return F.softplus(out)


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

edge_t_to_t_cpu = edge_t_to_t.cpu().numpy()
edge_p_to_t_cpu = edge_p_to_t.cpu().numpy()

N_t = len(traffic_node_ids)
N_p = len(pollution_node_ids)


edge_p_to_t_offset = edge_p_to_t_cpu.copy()
edge_p_to_t_offset[0, :] += N_t

# Concatenate all edges
all_edges = np.concatenate([
    edge_t_to_t_cpu,
    edge_p_to_t_offset
], axis=1)

edge_index_unified = torch.tensor(all_edges, dtype=torch.long).to(device)

print(f"Traffic nodes: {N_t}")
print(f"Pollution nodes: {N_p}")
print(f"Total nodes: {N_t + N_p}")
print(f"Unified edge index shape: {edge_index_unified.shape}")


Using device: cpu
Traffic nodes: 48
Pollution nodes: 4
Total nodes: 52
Unified edge index shape: torch.Size([2, 404])


In [ ]:
# ==========================================
# SECTION 5: TRAINING & VALIDATION WITH METRICS
# ==========================================

model = HeteroSTGCN(
    N_t=N_t, N_p=N_p,
    F_t=X_traffic.shape[2],
    F_p=X_pollution.shape[2],
    hidden=64,
    K=3,
    gru_layers=2,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Traffic input features: {traffic_features}")

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(train_loader),
    epochs=60,
    pct_start=0.1,
    anneal_strategy="cos",
)
criterion = nn.HuberLoss(delta=30.0)

train_losses, val_losses = [], []
best_val_loss    = float("inf")
patience_counter = 0
EARLY_STOP_PATIENCE = 12
n_epochs = 20


def masked_loss(criterion, pred, target):
    valid = ~torch.isnan(target)
    if not valid.any():
        return torch.tensor(0.0, requires_grad=True, device=pred.device)
    return criterion(pred[valid], target[valid])


print("=" * 70)
print("Training HeteroSTGCN (per-node GRU + GAT spatial + lagged features)...")
print("=" * 70)

for epoch in range(n_epochs):
    # --- Training ---
    model.train()
    train_loss, train_count = 0.0, 0
    for xt_seq, xp_seq, y_true in train_loader:
        optimizer.zero_grad()
        xt_seq = xt_seq.to(device)
        xp_seq = xp_seq.to(device)
        y_true = y_true.to(device)
        pred   = model(xt_seq, xp_seq, edge_index_unified)
        loss   = masked_loss(criterion, pred, y_true)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        optimizer.step()
        scheduler.step()
        train_loss  += loss.item() * xt_seq.size(0)
        train_count += xt_seq.size(0)

    train_loss_avg  = train_loss / max(1, train_count)
    train_rmse = np.sqrt(train_loss_avg)
    train_losses.append(train_loss_avg)

    # --- Validation ---
    model.eval()
    val_loss, val_count = 0.0, 0
    with torch.no_grad():
        for xt_seq, xp_seq, y_true in val_loader:
            xt_seq = xt_seq.to(device)
            xp_seq = xp_seq.to(device)
            y_true = y_true.to(device)
            pred   = model(xt_seq, xp_seq, edge_index_unified)
            loss   = masked_loss(criterion, pred, y_true)
            val_loss  += loss.item() * xt_seq.size(0)
            val_count += xt_seq.size(0)

    val_loss_avg = val_loss / max(1, val_count)
    val_rmse  = np.sqrt(val_loss_avg)
    val_losses.append(val_loss_avg)

    if val_loss_avg < best_val_loss:
        best_val_loss    = val_loss_avg
        patience_counter = 0
        torch.save(model.state_dict(), "best_hetero_model.pt")
    else:
        patience_counter += 1

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1:02d}/{n_epochs} | "
          f"Train RMSE: {train_rmse:.2f} | Val RMSE: {val_rmse:.2f} | "
          f"LR: {lr_now:.2e} | patience: {patience_counter}/{EARLY_STOP_PATIENCE}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping at epoch {epoch+1}.")
        break

print(f"\nBest val RMSE ≈ {np.sqrt(best_val_loss):.2f}")

Model parameters: 89,069
Traffic input features: ['route', 'weekday', 'distance_to_center_km', 'traffic_intensity', 'speed(mph)', 'miles', 'avtime', 'TravelTime', 'hour', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'count_observed', 'count_lag1', 'count_lag2', 'count_lag3', 'count_lag24']
Training HeteroSTGCN (per-node GRU + GAT spatial + lagged features)...
Epoch 01/10 | Train RMSE: 78.73 | Val RMSE: 77.98 | LR: 5.29e-05 | patience: 0/12
Epoch 02/10 | Train RMSE: 78.72 | Val RMSE: 77.97 | LR: 1.43e-04 | patience: 0/12


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load('best_hetero_model.pt'))
model.eval()

test_loss = 0.0
test_count = 0
all_preds = []
all_targets = []
with torch.no_grad():
    for xt_seq, xp_seq, y_true in test_loader:
        xt_seq = xt_seq.to(device)
        xp_seq = xp_seq.to(device)
        y_true = y_true.to(device)
        
        pred = model(xt_seq, xp_seq, edge_index_unified)
        loss = masked_loss(criterion, pred, y_true)  # NaN-safe
        pred = pred.cpu().numpy().astype(int)

        test_loss += loss.item() * xt_seq.size(0)
        test_count += xt_seq.size(0)
        
        all_preds.append(pred)
        all_targets.append(y_true.cpu().numpy())

test_mse = test_loss / max(1, test_count)
test_rmse = np.sqrt(test_mse)

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

print(f"\nTest Results:")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Predictions shape: {all_preds.shape}")
print(f"Targets shape: {all_targets.shape}")


Test Results:
Test RMSE: 71.0297
Predictions shape: (48, 48)
Targets shape: (48, 48)


In [ ]:
# ==========================================
# SECTION 6: VISUALIZATION & EVALUATION METRICS
# ==========================================
import numpy as np
import plotly.graph_objects as go

# --- 1. PLOT TRAINING CURVES ---
fig_train = go.Figure()

rt_train_losses = np.sqrt(train_losses)
rt_val_losses = np.sqrt(val_losses)

fig_train.add_trace(go.Scatter(
    y=rt_train_losses, name='Train RMSE', mode='lines', line=dict(color='blue')
))
fig_train.add_trace(go.Scatter(
    y=rt_val_losses, name='Val RMSE', mode='lines', line=dict(color='red')
))
fig_train.update_layout(
    title='Training and Validation Loss (Heterogeneous Graph)',
    xaxis_title='Epoch',
    yaxis_title='RMSE Loss',
    hovermode='x unified',
    width=900, height=500
)
fig_train.show()


# --- 2. PLOT PREDICTIONS VS TARGETS FOR A SPECIFIC NODE ---
target_node_id = 23

node_targets = all_targets[:, target_node_id]
node_preds = all_preds[:, target_node_id]

sample_time_idx = slice(0, min(100, len(node_targets)))

fig_node_pred = go.Figure()
fig_node_pred.add_trace(
    go.Scatter(x=np.arange(len(node_targets[sample_time_idx])), y=node_targets[sample_time_idx],
               name='True Target', mode='lines', line=dict(color='blue', width=2))
)
fig_node_pred.add_trace(
    go.Scatter(x=np.arange(len(node_preds[sample_time_idx])), y=node_preds[sample_time_idx],
               name='Model Prediction', mode='lines', line=dict(color='red', width=2, dash='dash'))
)
fig_node_pred.update_layout(
    title=dict(text=f"Traffic Prediction vs Target Timeline (Node Index: {target_node_id})", font=dict(size=16)),
    xaxis_title='Test Timestep Index',
    yaxis_title='Traffic Count Volume',
    hovermode='x unified',
    height=500, width=900
)
fig_node_pred.show()


# --- 3. SEPARATED ERROR DISTRIBUTION (All Nodes Global Residuals) ---
errors = np.abs(all_preds - all_targets).flatten()

fig_err_dist = go.Figure()
fig_err_dist.add_trace(
    go.Histogram(x=errors, nbinsx=40, name='Absolute Residuals', marker_color='green', opacity=0.75)
)
fig_err_dist.update_layout(
    title=dict(text="Global Absolute Error Distribution (All Stations & Timestamps)", font=dict(size=16)),
    xaxis_title='Absolute Magnitude of Residual Error',
    yaxis_title='Frequency Occurrence',
    height=500, width=900
)
fig_err_dist.show()


# --- 4. GLOBAL PREDICTIONS VS TARGETS TIMELINE (All Nodes Flattened) ---
sample_global_idx = slice(0, min(100, len(all_preds)))

fig_global_pred = go.Figure()
fig_global_pred.add_trace(
    go.Scatter(x=np.arange(all_targets[sample_global_idx].size), y=all_targets[sample_global_idx].flatten(),
               name='Target (Global)', mode='lines', line=dict(color='blue', width=1))
)
fig_global_pred.add_trace(
    go.Scatter(x=np.arange(all_preds[sample_global_idx].size), y=all_preds[sample_global_idx].flatten(),
               name='Prediction (Global)', mode='lines', line=dict(color='red', width=1, dash='dash'))
)
fig_global_pred.update_layout(
    title=dict(text="Global Predictions vs Targets (Flattened Node Horizon)", font=dict(size=16)),
    xaxis_title='Sample Index (Timesteps × Nodes)',
    yaxis_title='Value',
    hovermode='x unified',
    height=550, width=1100
)
fig_global_pred.show()


# --- 5. SUMMARY METRICS ---
mae = np.mean(errors)
rmse = np.sqrt(np.mean((all_preds - all_targets) ** 2))

# Compute global wMAPE
total_actual_traffic = np.sum(np.abs(all_targets))
wmape = np.sum(np.abs(all_targets - all_preds)) / max(1e-6, total_actual_traffic)

# Compute specific localized node-level metrics for transparency
node_errors = np.abs(node_preds - node_targets)
node_mae = np.mean(node_errors)
node_wmape = np.sum(node_errors) / max(1e-6, np.sum(np.abs(node_targets)))

print("\n" + "=" * 70)
print("FINAL TEST METRICS")
print("=" * 70)
print(f"Global System MAE:                {mae:.4f}")
print(f"Global System RMSE:               {rmse:.4f}")
print(f"Global System wMAPE:              {wmape:.4f}")
print("-" * 70)
print(f"Selected Node ({target_node_id}) Local MAE:     {node_mae:.4f}")
print(f"Selected Node ({target_node_id}) Local wMAPE:   {node_wmape:.4f}")
print("-" * 70)
print(f"Min System Prediction:            {all_preds.min():.4f}")
print(f"Max System Prediction:            {all_preds.max():.4f}")
print(f"Min System Target:                {all_targets.min():.4f}")
print(f"Max System Target:                {all_targets.max():.4f}")
print("=" * 70)


FINAL TEST METRICS
Global System MAE:                180.9119
Global System RMSE:               284.5495
Global System wMAPE:              0.9517
----------------------------------------------------------------------
Selected Node (23) Local MAE:     153.5833
Selected Node (23) Local wMAPE:   0.9409
----------------------------------------------------------------------
Min System Prediction:            10.0000
Max System Prediction:            13.0000
Min System Target:                0.0000
Max System Target:                1132.0000


In [ ]:
# ==========================================
# SECTION 7: PERMUTATION FEATURE IMPORTANCE
# ==========================================

def eval_rmse_on_loader(model, loader):
    model.eval()
    sq, n = 0.0, 0
    with torch.no_grad():
        for xt, xp, yt in loader:
            xt, xp, yt = xt.to(device), xp.to(device), yt.to(device)
            pred = model(xt, xp, edge_index_unified)
            valid = ~torch.isnan(yt)
            sq += ((pred[valid] - yt[valid]) ** 2).sum().item()
            n  += valid.sum().item()
    return float(np.sqrt(sq / max(1, n)))


baseline_rmse = eval_rmse_on_loader(model, val_loader)
print(f"Baseline val RMSE: {baseline_rmse:.4f}")

importance_rows = []
N_REPEATS = 3

for feat_idx, feat_name in enumerate(traffic_features):
    deltas = []
    for _ in range(N_REPEATS):
        sq, n = 0.0, 0
        model.eval()
        with torch.no_grad():
            for xt, xp, yt in val_loader:
                xt = xt.clone().to(device)
                xp, yt = xp.to(device), yt.to(device)
                # Shuffle this feature across the batch dimension
                perm = torch.randperm(xt.size(0), device=device)
                xt[:, :, :, feat_idx] = xt[perm, :, :, feat_idx]
                pred  = model(xt, xp, edge_index_unified)
                valid = ~torch.isnan(yt)
                sq   += ((pred[valid] - yt[valid]) ** 2).sum().item()
                n    += valid.sum().item()
        deltas.append(float(np.sqrt(sq / max(1, n))) - baseline_rmse)
    importance_rows.append({
        "feature":     feat_name,
        "delta_rmse":  float(np.mean(deltas)),
        "importance":  float(max(0.0, np.mean(deltas))),
    })

feature_importance_df = (
    pd.DataFrame(importance_rows)
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(feature_importance_df)

fig_fi = go.Figure(go.Bar(
    x=feature_importance_df["importance"],
    y=feature_importance_df["feature"],
    orientation="h",
    marker_color="teal",
))
fig_fi.update_layout(
    title="Traffic feature importance (permutation ΔRMSE on validation set)",
    xaxis_title="Mean RMSE increase when feature is shuffled",
    yaxis_title="Feature",
    height=max(350, 40 * len(traffic_features)),
    width=900,
)
fig_fi.show()


Baseline val RMSE: 308.2479


,feature,delta_rmse,importance
0,hour_sin,0.079215,0.079215
1,hour,0.059989,0.059989
2,count_lag1,0.059771,0.059771
3,count_lag2,0.054551,0.054551
4,count_lag24,0.037530,0.037530
5,count_lag3,0.033700,0.033700
6,weekday_sin,0.025914,0.025914
7,is_weekend,0.007024,0.007024
8,speed(mph),0.006101,0.006101
9,TravelTime,0.004332,0.004332


# imputation

In [ ]:
# Move edge_index_full to device here — device is guaranteed to be defined by this point
edge_index_full = torch.tensor(all_edges_full, dtype=torch.long).to(device)
print(f"Full edge index shape: {edge_index_full.shape}")

imputation_loader = DataLoader(imputation_dataset, batch_size=32, shuffle=False)

# Build (site_id, direction) map in the same order as traffic_node_ids_full
# Node keys are "traffic_<site_id>_<direction>" e.g. "traffic_23_N"
# rsplit("_", 1) splits at the LAST underscore: ("traffic_23", "N")
traffic_node_map = []
for node_id in traffic_node_ids_full:
    node_key = node_id[len("traffic_"):]   # "N_23"
    direction, site_id_str = node_key.rsplit("_", 1)  # ("N", "23") — site_id second
    traffic_node_map.append((int(site_id_str), direction))

target_timestamps = aligned_index_full[imputation_dataset.seq_len:]
print(f"Imputation will cover {len(target_timestamps)} target timestamps across {len(traffic_node_map)} nodes.")


Full edge index shape: torch.Size([2, 404])
Imputation will cover 6938 target timestamps across 48 nodes.


In [ ]:
model.eval()

all_imputed_preds = []
with torch.no_grad():
    for xt_seq, xp_seq, _ in imputation_loader:
        xt_seq = xt_seq.to(device)
        xp_seq = xp_seq.to(device)
        pred = model(xt_seq, xp_seq, edge_index_full)
        all_imputed_preds.append(pred.cpu().numpy())

all_imputed_preds = np.concatenate(all_imputed_preds, axis=0)  # [T_pred, N_t_full]

imputed_count = 0

for seq_idx, ts in enumerate(target_timestamps):
    for node_idx, (site_id, direction) in enumerate(traffic_node_map):
        if site_id not in full_data or direction not in full_data[site_id]:
            continue
        df = full_data[site_id][direction]
        mask = (df["timestamp"] == ts) & df["count"].isna()
        if mask.any():
            # Round to nearest non-negative integer (car counts are discrete)
            value = int(max(0, round(float(all_imputed_preds[seq_idx, node_idx]))))
            df.loc[mask, "count"] = value
            df.loc[mask, "count_imputed"] = 1  # flag imputed rows in full_data directly
            imputed_count += int(mask.sum())

print(f"Imputation complete: {imputed_count:,} missing count values filled in full_data.")

# Quick sanity check
ref_site, ref_dir = traffic_node_map[0]
ref_df = full_data[ref_site][ref_dir]
n_obs = int((ref_df["count_imputed"] == 0).sum())
n_imp = int((ref_df["count_imputed"] == 1).sum())
obs_min = ref_df.loc[ref_df["count_imputed"] == 0, "count"].min()
obs_max = ref_df.loc[ref_df["count_imputed"] == 0, "count"].max()
imp_min = ref_df.loc[ref_df["count_imputed"] == 1, "count"].min()
imp_max = ref_df.loc[ref_df["count_imputed"] == 1, "count"].max()
print(f"Reference node ({ref_site}, {ref_dir}): {n_obs} observed, {n_imp} imputed.")
print(f"  Observed count range : [{obs_min:.0f}, {obs_max:.0f}]")
print(f"  Imputed  count range : [{imp_min:.0f}, {imp_max:.0f}]")


KeyboardInterrupt: 

## Save & visualize imputed counts

Imputation writes into the in-memory `full_data` dict (`full_data[site_id][direction]` DataFrames). Nothing is saved automatically — run the cells below to persist and plot results.

In [ ]:
from pathlib import Path

save_dir = Path("processed_data_pkl")
save_dir.mkdir(parents=True, exist_ok=True)

imputed_pkl_path = save_dir / "full_data_imputed.pkl"
with open(imputed_pkl_path, "wb") as f:
    pickle.dump(full_data, f)

export_dir = Path("imputed_data")
export_dir.mkdir(parents=True, exist_ok=True)

long_rows = []
for site_id, dir_dict in full_data.items():
    for compass_dir, df in dir_dict.items():
        part = df[["timestamp", "count", "count_imputed"]].copy()
        part["site_id"] = site_id
        part["direction"] = compass_dir
        long_rows.append(part)

imputed_counts_long = pd.concat(long_rows, ignore_index=True)
csv_path = export_dir / "imputed_counts_long.csv"
imputed_counts_long.to_csv(csv_path, index=False)

remaining_na = int(imputed_counts_long["count"].isna().sum())
n_imputed_rows = int(imputed_counts_long["count_imputed"].sum())
print(f"Pickle (nested dict): {imputed_pkl_path.resolve()}")
print(f"Long CSV:             {csv_path.resolve()}  ({len(imputed_counts_long):,} rows)")
print(f"Total imputed rows in long CSV: {n_imputed_rows:,}")
if remaining_na:
    print(f"Warning: {remaining_na:,} count values are still NaN.")
else:
    print("All count values are non-null in the exported long table.")


Pickle (nested dict): C:\Users\lucch\Desktop\thesis\processed_data_pkl\full_data_imputed.pkl
Long CSV:             C:\Users\lucch\Desktop\thesis\imputed_data\imputed_counts_long.csv  (333,600 rows)
Total imputed rows in long CSV: 316,910


In [ ]:
# Plot imputed hourly counts for one site / direction
viz_site = 23
viz_direction = "N"
window = slice(0, 500)  # first N timesteps; use slice(None) for full series

series = full_data[viz_site][viz_direction].copy()
series["timestamp"] = pd.to_datetime(series["timestamp"])

fig_imputed = go.Figure()

# Full timeline (observed + imputed combined)
fig_imputed.add_trace(
    go.Scatter(
        x=series["timestamp"].iloc[window],
        y=series["count"].iloc[window],
        mode="lines",
        name="Count (full timeline)",
        line=dict(color="steelblue", width=1.5),
    )
)

# Highlight imputed points using count_imputed flag
imp_mask = series["count_imputed"] == 1
if imp_mask.any():
    imp_series = series[imp_mask].copy()
    fig_imputed.add_trace(
        go.Scatter(
            x=imp_series["timestamp"].iloc[window],
            y=imp_series["count"].iloc[window],
            mode="markers",
            name="Model-imputed",
            marker=dict(color="crimson", size=5, symbol="circle-open"),
        )
    )

# Observed counts from training data (may be a shorter window)
if viz_site in training_data and viz_direction in training_data[viz_site]:
    observed = training_data[viz_site][viz_direction][["timestamp", "count"]].copy()
    observed["timestamp"] = pd.to_datetime(observed["timestamp"])
    fig_imputed.add_trace(
        go.Scatter(
            x=observed["timestamp"],
            y=observed["count"],
            mode="markers",
            name="Training observed",
            marker=dict(color="green", size=6),
        )
    )

fig_imputed.update_layout(
    title=f"Traffic count — site {viz_site}, direction {viz_direction}",
    xaxis_title="Timestamp",
    yaxis_title="Count",
    hovermode="x unified",
    width=1000,
    height=500,
)
fig_imputed.show()
